In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# ===== CHINH DUONG DAN O DAY =====
PARTITION_PATH = "/content/drive/MyDrive/ĐATN/data/raw/DeepFashion-MultiModal_sub/Eval Partition List.txt"   # file list_eval_partition.txt cua In-shop
CAPTIONS_PATH  = "/content/drive/MyDrive/ĐATN/data/raw/DeepFashion-MultiModal_sub/Captions.json"             # file captions cua DeepFashion-MultiModal
OUTPUT_DIR     = "/content/drive/MyDrive/ĐATN/data/processed/ground_truth_text"              # thu muc luu ket qua (tu dong tao)
# =================================

for p in (PARTITION_PATH, CAPTIONS_PATH):
    print(p, "| ton tai:", os.path.exists(p))
os.makedirs(OUTPUT_DIR, exist_ok=True)

/content/drive/MyDrive/ĐATN/data/raw/DeepFashion-MultiModal_sub/Eval Partition List.txt | ton tai: True
/content/drive/MyDrive/ĐATN/data/raw/DeepFashion-MultiModal_sub/Captions.json | ton tai: True


In [ ]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

IMG_RE = re.compile(r"^(MEN|WOMEN)-(.+)-(id_\d{8})-(\d+_\d+_[A-Za-z]+)\.jpg$")


def load_partition(path):
    lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    declared = int(lines[0].strip())
    header = lines[1].split()
    assert header == ["image_name", "item_id", "evaluation_status"], f"Header la: {header}"
    rows = [ln.split() for ln in lines[2:] if ln.strip()]
    assert all(len(r) == 3 for r in rows), "Co dong khong du 3 cot"
    df = pd.DataFrame(rows, columns=["key", "item_id", "split"])
    assert len(df) == declared, f"File khai bao {declared} dong nhung doc duoc {len(df)}"
    assert set(df["split"]) <= {"train", "query", "gallery"}, "Co gia tri split la"
    assert not df["key"].duplicated().any(), "Co duong dan anh trung trong partition"
    return df


part = load_partition(PARTITION_PATH)
print("So anh trong partition:", len(part))
print("So san pham (item_id):", part["item_id"].nunique())
print(part["split"].value_counts().to_string())

So anh trong partition: 52712
So san pham (item_id): 7982
split
train      25882
query      14218
gallery    12612


In [ ]:
with open(CAPTIONS_PATH, encoding="utf-8") as f:
    captions_raw = json.load(f)
assert isinstance(captions_raw, dict), "Captions.json phai la dict {ten_file: caption}"

cap = pd.DataFrame({
    "file_name": list(captions_raw.keys()),
    "caption": list(captions_raw.values()),
})
cap["row_in_captions"] = np.arange(len(cap))
cap["caption"] = cap["caption"].astype(str).str.strip()

is_empty = cap["caption"].eq("")
captions_removed_empty = cap[is_empty].copy()
cap = cap[~is_empty].reset_index(drop=True)

print("Tong so caption trong file:", len(captions_raw))
print("Caption rong bi loai      :", len(captions_removed_empty))
print("Caption con lai           :", len(cap))

Tong so caption trong file: 12694
Caption rong bi loai      : 0
Caption con lai           : 12694


In [ ]:
parsed = cap["file_name"].str.extract(IMG_RE)
bad = parsed.isna().any(axis=1)
assert not bad.any(), f"{bad.sum()} ten file khong parse duoc, vi du: {cap.loc[bad, 'file_name'].head(3).tolist()}"

cap["gender"] = parsed[0]
cap["category"] = parsed[1]
cap["parsed_item_id"] = parsed[2]
cap["file_stem"] = parsed[3]
cap["image_id"] = cap["file_name"].str.replace(r"\.jpg$", "", regex=True)
cap["view"] = cap["file_stem"].str.extract(r"^\d+_\d+_([A-Za-z]+)$")[0]
cap["key"] = ("img/" + cap["gender"] + "/" + cap["category"] + "/"
              + cap["parsed_item_id"] + "/" + cap["file_stem"] + ".jpg")

df = cap.merge(part, on="key", how="left", validate="one_to_one")

unmatched = df[df["split"].isna()].copy()
print("Anh khong khop partition:", len(unmatched))
if len(unmatched) > 0:
    unmatched.to_csv(os.path.join(OUTPUT_DIR, "unmatched_images.csv"), index=False)
    df = df[df["split"].notna()].copy()

assert (df["item_id"] == df["parsed_item_id"]).all(), "item_id trong ten file khac item_id partition"

print("Anh da anh xa thanh cong:", len(df))
print("So san pham:", df["item_id"].nunique())
print(df["split"].value_counts().to_string())

Anh khong khop partition: 0
Anh da anh xa thanh cong: 12694
So san pham: 6899
split
train      6210
query      3541
gallery    2943


In [ ]:
query_all = df[df["split"] == "query"].copy()
gallery = df[df["split"] == "gallery"].copy()

assert set(query_all["image_id"]).isdisjoint(set(gallery["image_id"])), "Co anh vua query vua gallery"

gallery_items = set(gallery["item_id"])
has_positive = query_all["item_id"].isin(gallery_items)
queries = query_all[has_positive].copy()
queries_removed = query_all[~has_positive].copy()

pairs = (
    queries[["image_id", "item_id"]].rename(columns={"image_id": "query_image_id"})
    .merge(
        gallery[["image_id", "item_id"]].rename(columns={"image_id": "gallery_image_id"}),
        on="item_id", how="inner",
    )
    .sort_values(["query_image_id", "gallery_image_id"]).reset_index(drop=True)
)

qrels = {
    q: g["gallery_image_id"].tolist()
    for q, g in pairs.groupby("query_image_id", sort=True)
}

# kiem tra hop le
assert len(queries) > 0
assert set(qrels.keys()) == set(queries["image_id"]), "Co query khong co anh lien quan"
assert all(len(v) >= 1 for v in qrels.values())
item_of = dict(zip(df["image_id"], df["item_id"]))
assert all(item_of[p] == item_of[q] for q, ps in qrels.items() for p in ps), "Co cap khac item_id"
assert set(pairs["gallery_image_id"]) <= set(gallery["image_id"])

print("Query goc            :", len(query_all))
print("Query hop le         :", len(queries))
print("Query bi loai        :", len(queries_removed))
print("Gallery              :", len(gallery), "anh,", gallery["item_id"].nunique(), "san pham")
print("Cap query - gallery  :", len(pairs))

Query goc            : 3541
Query hop le         : 1893
Query bi loai        : 1648
Gallery              : 2943 anh, 2046 san pham
Cap query - gallery  : 4702


In [ ]:
n_rel = pairs.groupby("query_image_id").size()
view_of = dict(zip(df["image_id"], df["view"]))
cross_view = (pairs["query_image_id"].map(view_of) != pairs["gallery_image_id"].map(view_of)).mean()
n_words = df["caption"].str.split().str.len()

print("So anh lien quan moi query:")
print(n_rel.describe().round(2).to_string())
print("Ty le query chi co 1 anh lien quan:", round(float((n_rel == 1).mean()), 4))
print("Ty le cap khac goc chup           :", round(float(cross_view), 4))
print()
print("Goc chup cua query hop le:", queries["view"].value_counts().to_dict())
print("Goc chup cua gallery     :", gallery["view"].value_counts().to_dict())
print()
print("Do dai caption (so tu):", n_words.describe().round(1).to_dict())
print("Caption <= 10 tu:", int((n_words <= 10).sum()), "anh")

So anh lien quan moi query:
count    1893.00
mean        2.48
std         2.53
min         1.00
25%         1.00
50%         2.00
75%         3.00
max        15.00
Ty le query chi co 1 anh lien quan: 0.4945
Ty le cap khac goc chup           : 0.504

Goc chup cua query hop le: {'full': 693, 'additional': 551, 'front': 395, 'side': 151, 'back': 100, 'flat': 3}
Goc chup cua gallery     : {'full': 1489, 'front': 832, 'additional': 287, 'side': 194, 'back': 140, 'flat': 1}

Do dai caption (so tu): {'count': 12694.0, 'mean': 50.3, 'std': 10.1, 'min': 16.0, '25%': 43.0, '50%': 50.0, '75%': 57.0, 'max': 93.0}
Caption <= 10 tu: 0 anh


In [ ]:
cols = ["image_id", "item_id", "gender", "category", "view", "split",
        "row_in_captions", "caption"]

df.sort_values("row_in_captions")[cols].to_csv(
    os.path.join(OUTPUT_DIR, "mapping_full.csv"), index=False, encoding="utf-8-sig")
queries[cols].sort_values("row_in_captions").to_csv(
    os.path.join(OUTPUT_DIR, "queries.csv"), index=False, encoding="utf-8-sig")
gallery[cols].sort_values("row_in_captions").to_csv(
    os.path.join(OUTPUT_DIR, "gallery.csv"), index=False, encoding="utf-8-sig")
queries_removed[cols].sort_values("row_in_captions").to_csv(
    os.path.join(OUTPUT_DIR, "queries_removed_no_positive.csv"), index=False, encoding="utf-8-sig")
captions_removed_empty[["file_name", "row_in_captions"]].to_csv(
    os.path.join(OUTPUT_DIR, "captions_removed_empty.csv"), index=False, encoding="utf-8-sig")
pairs.to_csv(os.path.join(OUTPUT_DIR, "ground_truth_pairs.csv"), index=False, encoding="utf-8-sig")

with open(os.path.join(OUTPUT_DIR, "qrels.json"), "w", encoding="utf-8") as f:
    json.dump(qrels, f, ensure_ascii=False, indent=1)

report = {
    "partition_images": int(len(part)),
    "captions_in_file": int(len(captions_raw)),
    "captions_removed_empty": int(len(captions_removed_empty)),
    "images_mapped": int(len(df)),
    "items_mapped": int(df["item_id"].nunique()),
    "split_counts": {k: int(v) for k, v in df["split"].value_counts().items()},
    "queries_before_filter": int(len(query_all)),
    "queries_valid": int(len(queries)),
    "queries_removed_no_positive": int(len(queries_removed)),
    "gallery_images": int(len(gallery)),
    "gallery_items": int(gallery["item_id"].nunique()),
    "query_gallery_pairs": int(len(pairs)),
    "relevant_per_query_mean": round(float(n_rel.mean()), 3),
    "relevant_per_query_median": float(n_rel.median()),
    "relevant_per_query_max": int(n_rel.max()),
    "share_queries_with_single_positive": round(float((n_rel == 1).mean()), 4),
    "share_pairs_cross_view": round(float(cross_view), 4),
}
with open(os.path.join(OUTPUT_DIR, "report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Da luu vao:", OUTPUT_DIR)
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", name, f"({os.path.getsize(os.path.join(OUTPUT_DIR, name)) / 1024:.0f} KB)")

Da luu vao: /content/drive/MyDrive/ĐATN/data/processed/ground_truth_text
 - captions_removed_empty.csv (0 KB)
 - gallery.csv (1064 KB)
 - ground_truth_pairs.csv (420 KB)
 - mapping_full.csv (4584 KB)
 - qrels.json (296 KB)
 - queries.csv (672 KB)
 - queries_removed_no_positive.csv (603 KB)
 - report.json (1 KB)


In [ ]:
q2 = pd.read_csv(os.path.join(OUTPUT_DIR, "queries.csv"))
g2 = pd.read_csv(os.path.join(OUTPUT_DIR, "gallery.csv"))
with open(os.path.join(OUTPUT_DIR, "qrels.json"), encoding="utf-8") as f:
    qrels2 = json.load(f)

gid = set(g2["image_id"])
item_q = dict(zip(q2["image_id"], q2["item_id"]))
item_g = dict(zip(g2["image_id"], g2["item_id"]))

assert set(qrels2) == set(q2["image_id"]), "qrels va queries.csv khong khop"
assert all(set(v) <= gid for v in qrels2.values()), "qrels tro toi anh khong co trong gallery"
assert all(item_g[p] == item_q[q] for q, v in qrels2.items() for p in v), "Co cap khac item_id"
assert set(q2["image_id"]).isdisjoint(gid), "Query va gallery giao nhau"
assert q2["caption"].notna().all() and g2["caption"].notna().all(), "Co caption rong"
print("Kiem tra doc lai file: dat")
print("Query:", len(q2), "| Gallery:", len(g2), "| Cap:", sum(len(v) for v in qrels2.values()))

Kiem tra doc lai file: dat
Query: 1893 | Gallery: 2943 | Cap: 4702
